# Qwen3-8B Answer Evaluation (Validation)

## Purpose

This notebook evaluates the answers generated by `08_qwen_rag.ipynb` on the **validation** split, for each of the three retrievers (TF-IDF, BM25, dense). The goal is to compare retrievers *before* the test set is touched: the test set is only run once a retriever/configuration has been chosen from these validation results.


## Metrics

Since answers are free-form (not extractive spans), three complementary metrics are used:

- **Exact Match (EM):** 1 if the normalized generated answer equals the normalized reference answer, else 0. Expected to be low/near-zero for free-form answers; kept for completeness.
- **Token F1:** harmonic mean of precision and recall of overlapping tokens between the generated and reference answer (SQuAD-style).
- **ROUGE-L F1:** based on the longest common subsequence between the generated and reference answer, capturing overlap in order as well as content.

All three are computed per question and then averaged per retriever.

In [26]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import altair as alt

In [27]:
GENERATOR_NAME = "qwen"

RETRIEVERS = ("tfidf", "bm25", "dense")

SPLIT = "validation"

PROJECT_ROOT = Path.cwd()

GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "generation"
    / "qwen_colab"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "qwen"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert GENERATION_DIR.exists(), f"Ne postoji: {GENERATION_DIR}"

print("Generisani odgovori:", GENERATION_DIR)
print("Rezultati evaluacije:", RESULTS_DIR)
print("Split koji se evaluira:", SPLIT)

Generisani odgovori: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/generation/qwen_colab
Rezultati evaluacije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/qwen
Split koji se evaluira: validation


In [28]:
def load_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

## Loading Generated Answers


In [29]:
required_fields = {
    "question_id",
    "question",
    "answer",
    "retriever",
    "generator",
    "split",
    "generated_answer",
}

generation_data = {}

for retriever in RETRIEVERS:
    path = GENERATION_DIR / f"{retriever}_{SPLIT}_answers.jsonl"

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    for record in records:
        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

    generation_data[retriever] = records
    print(f"{retriever}/{SPLIT}: {len(records)} generisanih odgovora")

if not generation_data:
    raise FileNotFoundError(
        f"Nisu pronađeni generisani odgovori za split '{SPLIT}' ni za jedan retriever."
    )

tfidf/validation: 21 generisanih odgovora
bm25/validation: 21 generisanih odgovora
dense/validation: 21 generisanih odgovora


## Text Normalization and Metrics


In [30]:
TOKEN_PATTERN = re.compile(r"[\w]+", re.UNICODE)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str):
    return TOKEN_PATTERN.findall(normalize_text(text))

In [31]:
def compute_exact_match(prediction: str, reference: str) -> int:
    return int(normalize_text(prediction) == normalize_text(reference))


def compute_token_f1(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    from collections import Counter

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    overlap = sum(
        min(pred_counts[token], ref_counts[token])
        for token in pred_counts
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [32]:
def longest_common_subsequence_length(a: list, b: list) -> int:
    previous_row = [0] * (len(b) + 1)

    for token_a in a:
        current_row = [0] * (len(b) + 1)

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current_row[j] = previous_row[j - 1] + 1
            else:
                current_row[j] = max(previous_row[j], current_row[j - 1])

        previous_row = current_row

    return previous_row[-1]


def compute_rouge_l(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(pred_tokens, ref_tokens)

    if lcs_length == 0:
        return 0.0

    precision = lcs_length / len(pred_tokens)
    recall = lcs_length / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

## Per-Question and Aggregate Metrics

In [33]:
def evaluate_generations(records: list, retriever: str) -> pd.DataFrame:
    rows = []

    for record in records:
        prediction = record["generated_answer"]
        reference = record["answer"]

        rows.append({
            "question_id": record["question_id"],
            "retriever": retriever,
            "split": record["split"],
            "EM": compute_exact_match(prediction, reference),
            "F1": compute_token_f1(prediction, reference),
            "ROUGE_L": compute_rouge_l(prediction, reference),
        })

    return pd.DataFrame(rows)


per_question_frames = [
    evaluate_generations(records, retriever)
    for retriever, records in generation_data.items()
]

per_question_df = pd.concat(per_question_frames, ignore_index=True)

per_question_df.head()

,question_id,retriever,split,EM,F1,ROUGE_L
0,132,tfidf,validation,0,0.263736,0.197802
1,125,tfidf,validation,0,0.160920,0.114943
2,84,tfidf,validation,0,0.133333,0.100000
3,141,tfidf,validation,0,0.235294,0.176471
4,114,tfidf,validation,0,0.306667,0.160000


In [34]:
metrics_df = (
    per_question_df
    .groupby("retriever")[["EM", "F1", "ROUGE_L"]]
    .mean()
    .reset_index()
    .sort_values("F1", ascending=False)
)

metrics_df

,retriever,EM,F1,ROUGE_L
1,dense,0.0,0.280126,0.202783
0,bm25,0.0,0.269089,0.201277
2,tfidf,0.0,0.252122,0.192527


## Comparing Retrievers on Validation

In [35]:
metrics_plot_df = metrics_df.melt(
    id_vars="retriever",
    value_vars=["EM", "F1", "ROUGE_L"],
    var_name="metric",
    value_name="score",
)

alt.Chart(metrics_plot_df).mark_bar().encode(
    x=alt.X("retriever:N", title="Retriever"),
    y=alt.Y("score:Q", title="Score", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("metric:N", title="Metrika"),
    xOffset="metric:N",
    tooltip=["retriever", "metric", alt.Tooltip("score:Q", format=".3f")],
).properties(
    title=f"Qwen3-8B — poređenje retrievera na {SPLIT} skupu",
    width=500,
    height=350,
)

alt.Chart(...)

## Inspecting Individual Answers


In [36]:
best_retriever = metrics_df.iloc[0]["retriever"]

sample_df = (
    per_question_df[per_question_df["retriever"] == best_retriever]
    .sort_values("F1")
    .head(5)
)

sample_records = {
    record["question_id"]: record
    for record in generation_data[best_retriever]
}

for _, row in sample_df.iterrows():
    record = sample_records[row["question_id"]]
    print(f"Pitanje: {record['question']}")
    print(f"Referentni odgovor: {record['answer']}")
    print(f"Generisani odgovor: {record['generated_answer']}")
    print(f"EM={row['EM']:.0f}  F1={row['F1']:.3f}  ROUGE_L={row['ROUGE_L']:.3f}")
    print("-" * 80)

Pitanje: Objasniti detaljnije profajliranje grana u verifikaciji softvera.
Referentni odgovor: Profajliranje grana beleži koje grane uslovnog toka su izvršene i koliko puta, pa pokazuje koje alternative programa se zaista koriste tokom izvršavanja.
Generisani odgovor: Nema dovoljno informacija.
EM=0  F1=0.000  ROUGE_L=0.000
--------------------------------------------------------------------------------
Pitanje: Kako definišemo odnose pokrivenosti kod bele kutije?
Referentni odgovor: Različite vrste pokrivenosti imaju odnose u kojima jača pokrivenost podrazumeva određenu slabiju pokrivenost, ali obrnuto ne mora da važi. Odnos zavisi od kriterijuma koji se posmatraju.
Generisani odgovor: Pokrivenost kod bele kutije se definiše kao odnos broja izvršenih elemenata koda (naredbi, grananja, petlji itd.) u toku testiranja i ukupnog broja tih elemenata u kodu, množen sa 100%. Ova metrika se koristi za ocenu koliko dobro su test primeri pokrivali unutrašnju strukturu i logiku koda.
EM=0  F1=0.

In [37]:
per_question_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_per_question_metrics.csv",
    index=False,
)

metrics_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metrics.csv",
    index=False,
)

metadata = {
    "generator": GENERATOR_NAME,
    "split": SPLIT,
    "retrievers_evaluated": list(generation_data.keys()),
    "metrics": ["EM", "F1", "ROUGE_L"],
    "n_questions_per_retriever": {
        retriever: len(records)
        for retriever, records in generation_data.items()
    },
}

with (RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Sačuvani rezultati evaluacije u:", RESULTS_DIR)

Sačuvani rezultati evaluacije u: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/qwen
